# 06 — Practice: Module 1 — Core Foundations

25 interview-style questions covering Series, DataFrame, Indexing, Boolean Masking, and Data Types.
Each question has a complete, working solution.

In [ ]:
import pandas as pd
import numpy as np

# Shared dataset — used throughout this notebook
np.random.seed(42)
n = 200

employees = pd.DataFrame({
    'emp_id':           range(1001, 1001 + n),
    'name':             [f'Employee_{i:03d}' for i in range(n)],
    'department':       np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', 'Finance'], n),
    'salary':           np.random.randint(40000, 150000, n).astype(float),
    'experience_years': np.random.randint(1, 20, n),
    'join_date':        pd.date_range('2010-01-01', periods=n, freq='W'),
    'rating':           np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], n),
    'city':             np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai'], n),
    'active':           np.random.choice([True, False], n, p=[0.85, 0.15])
})

# Introduce some NaN
nan_idx = np.random.choice(n, 10, replace=False)
employees.loc[nan_idx, 'salary'] = np.nan

print(f"Shape: {employees.shape}")
employees.head(3)

---
### Q1: Series alignment — what does `s1 + s2` produce with mismatched indexes?

In [ ]:
s1 = pd.Series({'Alice': 90000, 'Bob': 85000, 'Carol': 110000})
s2 = pd.Series({'Bob': 5000, 'Carol': 8000, 'Dave': 3000})

# Labels align: Bob and Carol match. Alice and Dave are unmatched → NaN
print("s1 + s2 (default — NaN for unmatched):")
print(s1 + s2)

# Fix: use fill_value=0
print("\ns1.add(s2, fill_value=0):")
print(s1.add(s2, fill_value=0))

---
### Q2: Find the second highest salary without sorting

In [ ]:
salary = employees['salary'].dropna()

# Method 1: nlargest(2) — O(n log k) — more efficient than full sort
second_highest = salary.nlargest(2).iloc[-1]
print(f"Second highest (nlargest): {second_highest:,.0f}")

# Method 2: unique + sort
second_highest_v2 = sorted(salary.unique(), reverse=True)[1]
print(f"Second highest (unique+sort): {second_highest_v2:,.0f}")

---
### Q3: Rename columns using a mapping dict

In [ ]:
rename_map = {
    'experience_years': 'exp_yrs',
    'join_date': 'start_date',
    'active': 'is_active'
}

renamed = employees.rename(columns=rename_map)
print(renamed.columns.tolist())

---
### Q4: Select rows where salary > 100000 AND department is Engineering using loc

In [ ]:
mask = (
    (employees['salary'] > 100000) &
    (employees['department'] == 'Engineering')
)

result = employees.loc[mask, ['name', 'department', 'salary', 'experience_years']]
print(f"Rows: {len(result)}")
print(result.head())

---
### Q5: Use iloc to select the last 3 rows and first 2 columns

In [ ]:
# iloc uses position-based indexing; negative works for rows
result = employees.iloc[-3:, :2]
print(result)

---
### Q6: MultiIndex DataFrame — select data using xs()

In [ ]:
midx = pd.MultiIndex.from_product(
    [['Engineering', 'Sales', 'Finance'],
     ['Mumbai', 'Delhi']],
    names=['department', 'city']
)
np.random.seed(0)
sales_df = pd.DataFrame(
    np.random.randint(100000, 900000, (6, 2)),
    index=midx, columns=['Q1', 'Q2']
)

# xs() — cross-section — select all departments in Delhi
print("All departments in Delhi:")
print(sales_df.xs('Delhi', level='city'))

# xs() — select Engineering, all cities
print("\nAll cities for Engineering:")
print(sales_df.xs('Engineering', level='department'))

---
### Q7: IQR outlier detection and removal

In [ ]:
salary = employees['salary'].dropna()

Q1, Q3 = salary.quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

clean = employees.loc[
    employees['salary'].between(lower, upper) | employees['salary'].isna()
]
print(f"Original: {len(employees)}  Clean: {len(clean)}")
print(f"Removed:  {len(employees) - len(clean)} outliers")

---
### Q8: Use query() with a local variable

In [ ]:
min_salary = 90000
target_cities = ['Mumbai', 'Bangalore']

result = employees.query(
    "salary > @min_salary and city in @target_cities and active == True"
)
print(f"Filtered: {len(result)} employees")
print(result[['name', 'city', 'salary']].head())

---
### Q9: Convert object column to category — show memory saved

In [ ]:
mem_obj = employees['department'].memory_usage(deep=True)
mem_cat = employees['department'].astype('category').memory_usage(deep=True)

print(f"Object:   {mem_obj:,} bytes")
print(f"Category: {mem_cat:,} bytes")
print(f"Saved:    {(1 - mem_cat/mem_obj):.1%}")

---
### Q10: Add salary_adjusted — 1.5x for Sales, original otherwise (no loops)

In [ ]:
employees['salary_adjusted'] = np.where(
    employees['department'] == 'Sales',
    employees['salary'] * 1.5,
    employees['salary']
)

print(
    employees[['department', 'salary', 'salary_adjusted']]
    .groupby('department')
    .mean()
    .round(0)
)

---
### Q11: Find all employees in multiple departments using isin()

In [ ]:
tech_depts = ['Engineering', 'Finance']
tech_employees = employees[employees['department'].isin(tech_depts)]
print(f"Tech employees: {len(tech_employees)}")
print(tech_employees['department'].value_counts())

---
### Q12: Demonstrate SettingWithCopyWarning — cause and fix

In [ ]:
# CAUSE: Chained indexing on a filtered DataFrame
# This may or may not modify the original — undefined behavior!
# employees[employees['department'] == 'HR']['salary'] = 50000  # DANGEROUS

# FIX 1: Use loc with a single operation
employees_copy = employees.copy()
employees_copy.loc[employees_copy['department'] == 'HR', 'salary'] = 50000
print("HR salaries set to 50000:")
print(employees_copy[employees_copy['department'] == 'HR']['salary'].unique())

# FIX 2: Explicit copy before modifying a slice
hr_df = employees[employees['department'] == 'HR'].copy()  # .copy() avoids warning
hr_df['salary'] = 50000

---
### Q13: Use pd.cut() to bin salaries into Low/Mid/High

In [ ]:
salary_clean = employees.dropna(subset=['salary']).copy()

salary_clean['salary_band'] = pd.cut(
    salary_clean['salary'],
    bins=[0, 65000, 100000, 200000],
    labels=['Low', 'Mid', 'High'],
    right=True
)
print(salary_clean['salary_band'].value_counts().sort_index())

---
### Q14: Difference between df.values and df.to_numpy() with nullable integers

In [ ]:
df_nullable = pd.DataFrame({
    'a': pd.array([1, 2, None], dtype='Int64'),
    'b': pd.array([4, 5, 6],    dtype='Int64')
})

# .values: may not handle nullable types correctly — returns object array
print(".values dtype:", df_nullable.values.dtype)      # likely object

# .to_numpy(): preferred — handles nullable, allows specifying dtype/na_value
arr = df_nullable.to_numpy(dtype='float64', na_value=np.nan)
print(".to_numpy() dtype:", arr.dtype)  # float64
print(arr)

---
### Q15: Filter employees who joined between 2015–2020

In [ ]:
mask = employees['join_date'].between(
    pd.Timestamp('2015-01-01'),
    pd.Timestamp('2020-12-31')
)
joined_2015_2020 = employees[mask]
print(f"Joined 2015-2020: {len(joined_2015_2020)} employees")
print(joined_2015_2020['join_date'].dt.year.value_counts().sort_index())

---
### Q16: select_dtypes + correlation matrix

In [ ]:
numeric = employees.select_dtypes(include='number')
corr = numeric.corr().round(3)
print(corr)

---
### Q17: Prove that fancy indexing returns a copy

In [ ]:
subset = employees[['name', 'salary']]  # fancy indexing → copy
original_val = employees.loc[0, 'salary']

subset.loc[0, 'salary'] = 999999  # modify copy

print(f"Original employees.salary[0]: {employees.loc[0, 'salary']}")
print(f"subset.salary[0]:             {subset.loc[0, 'salary']}")
print("They differ — fancy indexing gives a copy, not a view!")

---
### Q18: Replace all salaries below 50000 with 50000 using clip()

In [ ]:
original_min = employees['salary'].min()
clipped = employees['salary'].clip(lower=50000)

print(f"Original min: {original_min:,.0f}")
print(f"Clipped min:  {clipped.min():,.0f}")
print(f"Values below 50000 after clip: {(clipped < 50000).sum()}")

---
### Q19: Find rows where ANY column has NaN

In [ ]:
rows_with_nan = employees[employees.isna().any(axis=1)]
print(f"Rows with any NaN: {len(rows_with_nan)}")
print(rows_with_nan[['name', 'salary']].head())

---
### Q20: Find employees from cities ending in 'i' using regex

In [ ]:
# Cities ending in 'i': Mumbai, Delhi, Chennai
mask = employees['city'].str.contains(r'i$', regex=True)
print(employees[mask]['city'].value_counts())

---
### Q21: Convert a column with mixed types to numeric with coerce

In [ ]:
messy = pd.Series(['85000', '90000', 'N/A', '75000', 'unknown', '110000'])

converted = pd.to_numeric(messy, errors='coerce')
print(converted)
print(f"\nMean (ignoring NaN): {converted.mean():,.0f}")

---
### Q22: Demonstrate loc slice is inclusive, iloc slice is exclusive

In [ ]:
df = pd.DataFrame({'val': range(10)}, index=list('abcdefghij'))

# loc: INCLUSIVE on both ends
print("loc['b':'e']:")
print(df.loc['b':'e'])  # b, c, d, e — 4 rows

# iloc: EXCLUSIVE on end
print("\niloc[1:5]:")
print(df.iloc[1:5])     # index 1,2,3,4 → b,c,d,e — 4 rows

print("\nBoth return 4 rows but loc end is included, iloc end is excluded")

---
### Q23: Create a categorical column with custom order and sort by it

In [ ]:
df = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'level': ['Senior', 'Junior', 'Mid', 'Executive', 'Junior']
})

level_order = ['Junior', 'Mid', 'Senior', 'Executive']
df['level'] = pd.Categorical(df['level'], categories=level_order, ordered=True)

sorted_df = df.sort_values('level')
print(sorted_df)

---
### Q24: Vectorized vs loop — speed comparison on 500K rows

In [ ]:
import time

big = pd.DataFrame({
    'salary': np.random.randint(40000, 150000, 500_000),
    'bonus_pct': np.random.uniform(0.05, 0.20, 500_000)
})

# Loop (iterrows)
start = time.perf_counter()
results_loop = []
for _, row in big.head(10000).iterrows():  # 10k for sanity
    results_loop.append(row['salary'] * (1 + row['bonus_pct']))
t_loop = time.perf_counter() - start

# Vectorized
start = time.perf_counter()
results_vec = (big['salary'] * (1 + big['bonus_pct'])).values
t_vec = time.perf_counter() - start

print(f"Loop (10k rows):    {t_loop*1000:.1f}ms")
print(f"Vectorized (500k): {t_vec*1000:.1f}ms")
print(f"Vectorized is ~{t_loop/t_vec * 50:.0f}x faster per row")

---
### Q25: Value counts with normalize — find department share of workforce

In [ ]:
dept_share = employees['department'].value_counts(normalize=True).mul(100).round(1)
dept_share.name = 'share_%'
print(dept_share.to_frame())

---
## Summary — Module 1 Key Takeaways

| Topic | Key Insight |
|-------|-------------|
| Series alignment | Operations align by **index label** — mismatches produce NaN |
| loc vs iloc | `loc` = labels (inclusive slices), `iloc` = positions (exclusive end) |
| Boolean masking | Use `&`, `|`, `~` — never Python `and/or` on Series |
| Chained indexing | Always use single `.loc[mask, col]` for writes |
| dtype optimization | `category` for low-cardinality strings saves 70-90% memory |
| Nullable int | Use `Int64` (capital I) when integers can have NaN |
| Vectorization | Always beats loops — 10-1000x faster |